# Milestone 5 — Context engineering for Ministral-3B

This notebook records the context-engineering ablation that followed the weak
Ministral-3B baseline. Schema grounding is the main intervention; general SQL
instructions and strict output-envelope normalization are also tested. The goal was not to tune on the frozen evaluation set.
It was to learn, using only the 12-question `working_dev` split, which domain
information helps a small model produce executable and correct SQLite.

**Checkpoint outcome:** relationships and descriptions mainly improved
executability; verified values fixed a small number of literal errors; a
compact context plus general SQL guardrails produced the best accuracy; strict
removal of one surrounding Markdown fence improved executability without a
second model call. A one-shot repair pass cost more tokens and did not improve
accuracy, so it was rejected.

The selected Milestone 5 candidate is therefore:

1. compact financial-domain context,
2. general SQL-construction guardrails, and
3. strict single-fence output normalization.

No frozen-evaluation question is inspected or executed in this notebook.

**Terminology:** a model harness is any surrounding invocation, validation, or repair logic; it is not limited to agents. Nevertheless, `context engineering` is the more precise name here because most experiments change what the model receives rather than introduce orchestration.


## Experiment map

The work evolved in seven deliberate steps:

| Stage | Question it answered |
|---|---|
| Bare schema | How weak is the original small-model baseline? |
| Relationships | Do explicit foreign keys reduce invented joins? |
| Column descriptions | Can the model decode opaque fields such as `A11` and `A13`? |
| Verified values | Do exact stored codes such as `M`, `OWNER`, and `POPLATEK TYDNE` help? |
| Compact context | Can we keep the useful facts while reducing prompt burden? |
| Guardrails + strict unwrap | Do general construction checks and deterministic envelope cleanup help? |
| One repair attempt | Is a second model call worth its added cost? |

Each model ablation used one attempt per working question. Reference SQL and
expected results were available only to the evaluator, never to the model.


## Local prerequisites

Run the notebook with `notebooks/` as the working directory. The large BIRD
archives are only temporary extraction material. The persistent local files
needed here are:

```text
data/
  financial.sqlite
  finch_dataset.json
  financial_metadata/database_description/
    account.csv card.csv client.csv disp.csv
    district.csv loan.csv order.csv trans.csv
  runs/                         # optional local raw run artifacts
```

The metadata CSVs are the eight files from BIRD's
`dev_databases/financial/database_description` directory. Their combined hash
is checked below before use. All of `data/` remains intentionally gitignored.


In [1]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import csv
import hashlib
import json
import os
import re
import sqlite3

from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible
from domain_adapt.frontier import load_baseline_config
from domain_adapt.slm import request_sql as request_slm_sql


DATA_DIR = Path("../data").resolve()
DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
METADATA_DIR = DATA_DIR / "financial_metadata" / "database_description"
MANIFEST_PATH = Path("../configs/eval_manifest_v1.json").resolve()
SLM_CONFIG_PATH = Path("../configs/slm_baseline_v1.json").resolve()

required_paths = (DB_PATH, DATASET_PATH, MANIFEST_PATH, SLM_CONFIG_PATH)
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing notebook prerequisites: {missing_paths}")

conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)
records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
slm_config = load_baseline_config(SLM_CONFIG_PATH)

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)

working_examples = sorted(
    (
        row for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in working_ids
    ),
    key=lambda row: row["question_id"],
)
assert len(working_examples) == 12


def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def sha256_text(value: str) -> str:
    return sha256(value.encode("utf-8")).hexdigest()


print("Working questions:", len(working_examples))
print("Frozen questions touched:", 0)


Working questions: 12
Frozen questions touched: 0


## 1. Make relationships explicit

Column names alone do not reliably tell a 3B model how tables connect. This
renderer uses SQLite's declared primary and foreign keys and adds a short
relationship list. The redundant-looking second representation is deliberate:
it makes join paths easy to find without introducing new information.


In [2]:
def build_relationship_schema_context(
    conn: sqlite3.Connection,
) -> str:
    """Render tables, primary keys, and declared relationships."""
    tables = [
        row[0]
        for row in conn.execute(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name
            """
        )
    ]

    context_lines = ["SQLite schema:"]

    all_relationships = []

    for table_name in tables:
        quoted_table = quote_identifier(table_name)

        columns = conn.execute(
            f"PRAGMA table_info({quoted_table})"
        ).fetchall()

        foreign_keys = conn.execute(
            f"PRAGMA foreign_key_list({quoted_table})"
        ).fetchall()

        entries = []

        for column in columns:
            column_name = column[1]
            declared_type = column[2] or "UNKNOWN"
            is_primary_key = column[5] > 0

            definition = (
                f"{quote_identifier(column_name)} "
                f"{declared_type}"
            )

            if is_primary_key:
                definition += " PRIMARY KEY"

            entries.append(definition)

        normalized_foreign_keys = sorted(
            (
                foreign_key[3],  # child column
                foreign_key[2],  # parent table
                foreign_key[4],  # parent column
            )
            for foreign_key in foreign_keys
        )

        for (
            child_column,
            parent_table,
            parent_column,
        ) in normalized_foreign_keys:
            entries.append(
                "FOREIGN KEY "
                f"({quote_identifier(child_column)}) "
                "REFERENCES "
                f"{quote_identifier(parent_table)} "
                f"({quote_identifier(parent_column)})"
            )

            all_relationships.append(
                (
                    table_name,
                    child_column,
                    parent_table,
                    parent_column,
                )
            )

        context_lines.append(
            f"\nCREATE TABLE {quoted_table} ("
        )

        for index, entry in enumerate(entries):
            comma = "," if index < len(entries) - 1 else ""
            context_lines.append(f"  {entry}{comma}")

        context_lines.append(");")

    # A compact second representation helps a small model find
    # the join path without changing the underlying information.
    context_lines.append("\nRelationships:")

    for (
        child_table,
        child_column,
        parent_table,
        parent_column,
    ) in sorted(all_relationships):
        context_lines.append(
            f"- {quote_identifier(child_table)}."
            f"{quote_identifier(child_column)} "
            "references "
            f"{quote_identifier(parent_table)}."
            f"{quote_identifier(parent_column)}"
        )

    return "\n".join(context_lines)


relationship_schema_context = (
    build_relationship_schema_context(conn)
)

relationship_schema_sha256 = sha256(
    relationship_schema_context.encode("utf-8")
).hexdigest()

print(relationship_schema_context)
print("\nRelationship-schema SHA-256:")
print(relationship_schema_sha256)


SQLite schema:

CREATE TABLE "account" (
  "account_id" INTEGER PRIMARY KEY,
  "district_id" INTEGER,
  "frequency" TEXT,
  "date" DATE,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "card" (
  "card_id" INTEGER PRIMARY KEY,
  "disp_id" INTEGER,
  "type" TEXT,
  "issued" DATE,
  FOREIGN KEY ("disp_id") REFERENCES "disp" ("disp_id")
);

CREATE TABLE "client" (
  "client_id" INTEGER PRIMARY KEY,
  "gender" TEXT,
  "birth_date" DATE,
  "district_id" INTEGER,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "disp" (
  "disp_id" INTEGER PRIMARY KEY,
  "client_id" INTEGER,
  "account_id" INTEGER,
  "type" TEXT,
  FOREIGN KEY ("account_id") REFERENCES "account" ("account_id"),
  FOREIGN KEY ("client_id") REFERENCES "client" ("client_id")
);

CREATE TABLE "district" (
  "district_id" INTEGER PRIMARY KEY,
  "A2" TEXT,
  "A3" TEXT,
  "A4" TEXT,
  "A5" TEXT,
  "A6" TEXT,
  "A7" TEXT,
  "A8" INTEGER,
  "A9" INTEGER,
  "A10"

In [3]:
EXPECTED_RELATIONSHIP_SCHEMA_SHA256 = (
    "26cdd17a25945a10041bda7e9bf23e83aaacb0c3f261a899b04f2d78ac2a7d89"
)

assert relationship_schema_sha256 == EXPECTED_RELATIONSHIP_SCHEMA_SHA256
assert relationship_schema_context.count("CREATE TABLE") == 8
print("Relationship schema hash:", relationship_schema_sha256)
print("Characters:", len(relationship_schema_context))


Relationship schema hash: 26cdd17a25945a10041bda7e9bf23e83aaacb0c3f261a899b04f2d78ac2a7d89
Characters: 2334


## 2. Load and validate the financial metadata

The official BIRD metadata is useful for column meanings, but it is not trusted
blindly. We first require exactly the same eight tables and columns as SQLite.
The only structural oddity is `account.csv`: the frequency-code explanation
overflowed into a blank CSV column. That value is intentionally excluded from
the descriptions-only context and later replaced with database-verified facts.


In [4]:
EXPECTED_METADATA_SHA256 = (
    "3f7be83f3f9bdba77d27a66ad38b1a81b3a40361168780f54ceedd985db5fc23"
)

description_csvs = sorted(METADATA_DIR.glob("*.csv"))
if len(description_csvs) != 8:
    raise FileNotFoundError(
        f"Expected 8 financial metadata CSVs in {METADATA_DIR}; "
        f"found {len(description_csvs)}"
    )

metadata_hash = hashlib.sha256()
metadata_by_table = {}

for path in description_csvs:
    raw_bytes = path.read_bytes()
    metadata_hash.update(path.name.encode("utf-8"))
    metadata_hash.update(b"\0")
    metadata_hash.update(raw_bytes)
    metadata_hash.update(b"\0")

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        metadata_by_table[path.stem] = list(csv.DictReader(file))

assert metadata_hash.hexdigest() == EXPECTED_METADATA_SHA256

database_tables = {
    row[0]
    for row in conn.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
    )
}
assert set(metadata_by_table) == database_tables

blank_overflow_records = []
for table_name, rows in metadata_by_table.items():
    sqlite_columns = [
        row[1]
        for row in conn.execute(
            f"PRAGMA table_info({quote_identifier(table_name)})"
        )
    ]
    metadata_columns = [
        (row.get("original_column_name") or "").strip()
        for row in rows
    ]
    assert [name.casefold() for name in metadata_columns] == [
        name.casefold() for name in sqlite_columns
    ]

    blank_overflow_records.extend(
        (table_name, row["original_column_name"], row.get("", ""))
        for row in rows
        if (row.get("") or "").strip()
    )

assert len(blank_overflow_records) == 1
assert blank_overflow_records[0][0:2] == ("account", "frequency")

print("Metadata tables validated:", len(metadata_by_table))
print("Financial metadata hash:", metadata_hash.hexdigest())
print("Blank overflow records classified:", len(blank_overflow_records))


Metadata tables validated: 8
Financial metadata hash: 3f7be83f3f9bdba77d27a66ad38b1a81b3a40361168780f54ceedd985db5fc23
Blank overflow records classified: 1


### What was safe to use from the metadata

Friendly names and column descriptions were useful: for example, they reveal
that `district.A11` means average salary and `district.A13` means 1996
unemployment. Several value descriptions were stale or misplaced, though:

- date fields were described as `YYMMDD`, while SQLite stores ten-character
  `YYYY-MM-DD` text;
- `loan.payments` was described with a month unit even though it is a payment
  amount;
- the two-letter bank-code note appeared under `order.account_to` rather than
  `order.bank_to`.

Therefore the descriptions-only prompt uses only `column_name` and
`column_description`. Formats and categorical values are verified against the
actual database before being added.


In [5]:
import hashlib
import re


def clean_metadata_text(value):
    return " ".join((value or "").strip().split())


def normalized_meaning(value):
    value = clean_metadata_text(value).casefold()
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def build_column_meanings(metadata_by_table):
    lines = ["Column meanings:"]

    for table_name in sorted(metadata_by_table):
        for row in metadata_by_table[table_name]:
            original_name = clean_metadata_text(
                row.get("original_column_name")
            )
            friendly_name = clean_metadata_text(
                row.get("column_name")
            )
            description = clean_metadata_text(
                row.get("column_description")
            )

            original_normalized = normalized_meaning(
                original_name.replace("_", " ")
            )

            meanings = []
            seen_meanings = set()

            for candidate in (friendly_name, description):
                candidate_normalized = normalized_meaning(candidate)

                if not candidate_normalized:
                    continue

                # The schema already exposes the original column name.
                if candidate_normalized == original_normalized:
                    continue

                # Avoid repeating identical friendly names/descriptions.
                if candidate_normalized in seen_meanings:
                    continue

                meanings.append(candidate)
                seen_meanings.add(candidate_normalized)

            if not meanings:
                continue

            quoted_table = table_name.replace('"', '""')
            quoted_column = original_name.replace('"', '""')

            lines.append(
                f'- "{quoted_table}"."{quoted_column}": '
                + "; ".join(meanings)
            )

    return "\n".join(lines)


column_meanings = build_column_meanings(metadata_by_table)

relationship_description_schema = (
    relationship_schema_context
    + "\n\n"
    + column_meanings
)

relationship_description_schema_hash = hashlib.sha256(
    relationship_description_schema.encode("utf-8")
).hexdigest()

print(relationship_description_schema)
print()
print(
    "Relationship + description schema SHA-256:",
    relationship_description_schema_hash,
)
print(
    "Schema characters:",
    len(relationship_description_schema),
)


SQLite schema:

CREATE TABLE "account" (
  "account_id" INTEGER PRIMARY KEY,
  "district_id" INTEGER,
  "frequency" TEXT,
  "date" DATE,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "card" (
  "card_id" INTEGER PRIMARY KEY,
  "disp_id" INTEGER,
  "type" TEXT,
  "issued" DATE,
  FOREIGN KEY ("disp_id") REFERENCES "disp" ("disp_id")
);

CREATE TABLE "client" (
  "client_id" INTEGER PRIMARY KEY,
  "gender" TEXT,
  "birth_date" DATE,
  "district_id" INTEGER,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "disp" (
  "disp_id" INTEGER PRIMARY KEY,
  "client_id" INTEGER,
  "account_id" INTEGER,
  "type" TEXT,
  FOREIGN KEY ("account_id") REFERENCES "account" ("account_id"),
  FOREIGN KEY ("client_id") REFERENCES "client" ("client_id")
);

CREATE TABLE "district" (
  "district_id" INTEGER PRIMARY KEY,
  "A2" TEXT,
  "A3" TEXT,
  "A4" TEXT,
  "A5" TEXT,
  "A6" TEXT,
  "A7" TEXT,
  "A8" INTEGER,
  "A9" INTEGER,
  "A10"

In [6]:
EXPECTED_DESCRIPTION_SCHEMA_SHA256 = (
    "86108a8314b1b09dcf48785fb8aac385c4bb0450a7d473b2f9bc1cf0c3f1a6a6"
)

assert relationship_description_schema_hash == EXPECTED_DESCRIPTION_SCHEMA_SHA256
for forbidden_fragment in (
    "POPLATEK MESICNE",
    "POPLATEK TYDNE",
    "POPLATEK PO OBRATU",
    "DISPONENT",
):
    assert forbidden_fragment not in relationship_description_schema

print("Relationship + descriptions hash:", relationship_description_schema_hash)
print("Characters:", len(relationship_description_schema))
print("Value leakage guard passed.")


Relationship + descriptions hash: 86108a8314b1b09dcf48785fb8aac385c4bb0450a7d473b2f9bc1cf0c3f1a6a6
Characters: 4900
Value leakage guard passed.


## 3. Verify values and formats against SQLite

Small models often translate English labels literally (`male`, `weekly`,
`running`) even when the database stores codes (`M`, `POPLATEK TYDNE`, `C` or
`D`). The following assertions establish the small, high-value glossary from
actual database contents rather than benchmark answers.


In [7]:
expected_categorical_values = {
    "account.frequency": {"POPLATEK MESICNE", "POPLATEK TYDNE", "POPLATEK PO OBRATU"},
    "card.type": {"classic", "junior", "gold"},
    "client.gender": {"M", "F"},
    "disp.type": {"OWNER", "DISPONENT"},
    "district.A3": {
        "south Moravia", "central Bohemia", "east Bohemia", "north Moravia",
        "north Bohemia", "west Bohemia", "south Bohemia", "Prague",
    },
    "loan.status": {"A", "B", "C", "D"},
}

for qualified_name, expected_values in expected_categorical_values.items():
    table_name, column_name = qualified_name.split(".")
    actual_values = {
        row[0]
        for row in conn.execute(
            f"SELECT DISTINCT {quote_identifier(column_name)} "
            f"FROM {quote_identifier(table_name)}"
        )
    }
    assert actual_values == expected_values

for table_name, column_name in (
    ("account", "date"),
    ("card", "issued"),
    ("client", "birth_date"),
    ("loan", "date"),
    ("trans", "date"),
):
    storage_types, minimum_length, maximum_length = conn.execute(
        f"SELECT GROUP_CONCAT(DISTINCT TYPEOF({quote_identifier(column_name)})), "
        f"MIN(LENGTH({quote_identifier(column_name)})), "
        f"MAX(LENGTH({quote_identifier(column_name)})) "
        f"FROM {quote_identifier(table_name)} "
        f"WHERE {quote_identifier(column_name)} IS NOT NULL"
    ).fetchone()
    assert storage_types == "text"
    assert (minimum_length, maximum_length) == (10, 10)

print("Verified categorical columns:", len(expected_categorical_values))
print("Verified date columns: 5 (TEXT, YYYY-MM-DD shape)")


Verified categorical columns: 6
Verified date columns: 5 (TEXT, YYYY-MM-DD shape)


In [8]:
verified_value_glossary = """Verified database values and formats:
- Use the exact stored text literals and casing shown below.
- All DATE columns are stored as TEXT in YYYY-MM-DD format.
- Monetary amount and balance columns use US dollars.
- "loan"."duration" is measured in months.
- "account"."frequency": 'POPLATEK MESICNE' = monthly issuance; 'POPLATEK TYDNE' = weekly issuance; 'POPLATEK PO OBRATU' = issuance after a transaction.
- "card"."type" uses 'junior', 'classic', or 'gold'.
- "client"."gender": 'M' = male; 'F' = female.
- "disp"."type": 'OWNER' = account owner; 'DISPONENT' = authorized user who is not the owner.
- "district"."A3" uses 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', or 'Prague'.
- "loan"."status": 'A' = finished with no problems; 'B' = finished but unpaid; 'C' = running and OK; 'D' = running with the client in debt.
- "order"."k_symbol": 'POJISTNE' = insurance; 'SIPO' = household payment; 'LEASING' = leasing; 'UVER' = loan payment.
- "trans"."type" uses 'PRIJEM', 'VYDAJ', or 'VYBER'; 'PRIJEM' means credit and 'VYDAJ' means withdrawal.
- "trans"."operation": 'VYBER KARTOU' = card withdrawal; 'VKLAD' = cash credit; 'PREVOD Z UCTU' = collection from another bank; 'VYBER' = cash withdrawal; 'PREVOD NA UCET' = remittance to another bank.
- "trans"."k_symbol": 'POJISTNE' = insurance; 'SLUZBY' = statement payment; 'UROK' = credited interest; 'SANKC. UROK' = sanction interest; 'SIPO' = household payment; 'DUCHOD' = pension; 'UVER' = loan payment.
"""


In [9]:
relationship_description_value_schema = (
    relationship_description_schema.rstrip()
    + "\n\n"
    + verified_value_glossary.strip()
)

relationship_description_value_schema_hash = hashlib.sha256(
    relationship_description_value_schema.encode("utf-8")
).hexdigest()
assert relationship_description_value_schema.count(
    "CREATE TABLE"
) == 8

assert "Column meanings:" in (
    relationship_description_value_schema
)

assert "Verified database values and formats:" in (
    relationship_description_value_schema
)

# Reject known stale or incorrect metadata.
for forbidden_fragment in (
    "YYMMDD",
    "account_to",
    "not useful",
    '"payments" is measured in months',
    "can only have the right",
):
    assert forbidden_fragment not in (
        verified_value_glossary
    )

# Confirm the important verified literals are present.
for required_fragment in (
    "'M' = male",
    "'POPLATEK TYDNE' = weekly issuance",
    "'C' = running and OK",
    "'D' = running with the client in debt",
    "'OWNER' = account owner",
    "'north Bohemia'",
    "YYYY-MM-DD",
):
    assert required_fragment in (
        verified_value_glossary
    )

print(verified_value_glossary)
print()
print(
    "Relationship + descriptions + values SHA-256:",
    relationship_description_value_schema_hash,
)
print(
    "Schema characters:",
    len(relationship_description_value_schema),
)
print("Verified-value safeguards passed.")
print("No model request made.")


Verified database values and formats:
- Use the exact stored text literals and casing shown below.
- All DATE columns are stored as TEXT in YYYY-MM-DD format.
- Monetary amount and balance columns use US dollars.
- "loan"."duration" is measured in months.
- "account"."frequency": 'POPLATEK MESICNE' = monthly issuance; 'POPLATEK TYDNE' = weekly issuance; 'POPLATEK PO OBRATU' = issuance after a transaction.
- "card"."type" uses 'junior', 'classic', or 'gold'.
- "client"."gender": 'M' = male; 'F' = female.
- "disp"."type": 'OWNER' = account owner; 'DISPONENT' = authorized user who is not the owner.
- "district"."A3" uses 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', or 'Prague'.
- "loan"."status": 'A' = finished with no problems; 'B' = finished but unpaid; 'C' = running and OK; 'D' = running with the client in debt.
- "order"."k_symbol": 'POJISTNE' = insurance; 'SIPO' = household payment; 'LEASING' = leasing; 'UVER' 

In [10]:
EXPECTED_VALUE_SCHEMA_SHA256 = (
    "ed139b776c279d00c25ce1a9247b41c13e294a8a6bb4a3f45053455286a790ae"
)
assert relationship_description_value_schema_hash == EXPECTED_VALUE_SCHEMA_SHA256


## 4. Compact the domain context

The fully expanded context reached 6,434 characters and increased average
prompt size without a proportional accuracy gain. The compact version below
retains every table, join path, useful opaque-column meaning, exact categorical
code, and date format in 3,628 characters—a 43.6% reduction.

This is a manual compression of database facts, not question-specific tuning.
Leakage guards reject question IDs, reference SQL, and stale metadata wording.


In [11]:
compact_domain_context = """SQLite financial database.

Schema:
- account: account_id INTEGER PK; district_id INTEGER FK->district.district_id; frequency TEXT; date DATE.
- card: card_id INTEGER PK; disp_id INTEGER FK->disp.disp_id; type TEXT; issued DATE.
- client: client_id INTEGER PK; gender TEXT; birth_date DATE; district_id INTEGER FK->district.district_id.
- disp: disp_id INTEGER PK; client_id INTEGER FK->client.client_id; account_id INTEGER FK->account.account_id; type TEXT.
- district: district_id INTEGER PK; A2 TEXT; A3 TEXT; A4 TEXT; A5 TEXT; A6 TEXT; A7 TEXT; A8 INTEGER; A9 INTEGER; A10 REAL; A11 INTEGER; A12 REAL; A13 REAL; A14 INTEGER; A15 INTEGER; A16 INTEGER.
- loan: loan_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; amount INTEGER; duration INTEGER; payments REAL; status TEXT.
- "order": order_id INTEGER PK; account_id INTEGER FK->account.account_id; bank_to TEXT; account_to INTEGER; amount REAL; k_symbol TEXT.
- trans: trans_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; type TEXT; operation TEXT; amount INTEGER; balance INTEGER; k_symbol TEXT; bank TEXT; account INTEGER.

Meanings:
- account.frequency = statement issuance frequency; account.date = account opening date.
- card.type = credit-card class; card.issued = card issue date.
- client.gender = client gender; client.birth_date = client birth date.
- disp.type = client's role for an account.
- district.A2 = district name; A3 = region; A4 = inhabitants; A5 = municipalities with fewer than 499 inhabitants; A6 = municipalities with 500-1999; A7 = municipalities with 2000-9999; A8 = municipalities with more than 10000; A10 = urban-inhabitant ratio; A11 = average salary; A12 = unemployment rate in 1995; A13 = unemployment rate in 1996; A14 = entrepreneurs per 1000 inhabitants; A15 = committed crimes in 1995; A16 = committed crimes in 1996.
- loan.date = approval date; loan.amount = approved amount; loan.duration = duration in months; loan.payments = monthly payment amount; loan.status = repayment status.
- "order".bank_to = recipient bank; account_to = recipient account; amount = debited amount; k_symbol = payment purpose.
- trans.date = transaction date; trans.type = credit or withdrawal; trans.operation = transaction mode; trans.amount = transaction amount; trans.balance = balance after the transaction; trans.k_symbol = transaction purpose; trans.bank = partner bank; trans.account = partner account.

Verified values and formats:
- Use exact stored text literals and casing.
- Dates are TEXT in YYYY-MM-DD format. Monetary amounts and balances use US dollars.
- account.frequency: 'POPLATEK MESICNE'=monthly; 'POPLATEK TYDNE'=weekly; 'POPLATEK PO OBRATU'=after transaction.
- card.type: 'junior', 'classic', 'gold'.
- client.gender: 'M'=male; 'F'=female.
- disp.type: 'OWNER'=owner; 'DISPONENT'=authorized non-owner.
- district.A3: 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', 'Prague'.
- loan.status: 'A'=finished without problems; 'B'=finished unpaid; 'C'=running and OK; 'D'=running in debt.
- "order".k_symbol: 'POJISTNE'=insurance; 'SIPO'=household; 'LEASING'=leasing; 'UVER'=loan.
- trans.type: 'PRIJEM'=credit; 'VYDAJ'=withdrawal; 'VYBER' is also stored.
- trans.operation: 'VYBER KARTOU'=card withdrawal; 'VKLAD'=cash credit; 'PREVOD Z UCTU'=collection from another bank; 'VYBER'=cash withdrawal; 'PREVOD NA UCET'=remittance to another bank.
- trans.k_symbol: 'POJISTNE'=insurance; 'SLUZBY'=statement payment; 'UROK'=credited interest; 'SANKC. UROK'=sanction interest; 'SIPO'=household; 'DUCHOD'=pension; 'UVER'=loan.
"""
compact_domain_context_hash = hashlib.sha256(
    compact_domain_context.encode("utf-8")
).hexdigest()

required_schema_fragments = (
    "account_id INTEGER PK",
    "district_id INTEGER FK->district.district_id",
    "disp_id INTEGER FK->disp.disp_id",
    "client_id INTEGER FK->client.client_id",
    "account_id INTEGER FK->account.account_id",
    "A11 = average salary",
    "A13 = unemployment rate in 1996",
    "trans.balance = balance after the transaction",
    "'M'=male",
    "'POPLATEK TYDNE'=weekly",
    "'C'=running and OK",
    "'D'=running in debt",
)

for fragment in required_schema_fragments:
    assert fragment in compact_domain_context

for forbidden_fragment in (
    "YYMMDD",
    "can only have the right",
    "Reference SQL",
    "Question ID",
):
    assert forbidden_fragment not in compact_domain_context

assert len(compact_domain_context) < len(
    relationship_description_value_schema
)

reduction = 1 - (
    len(compact_domain_context)
    / len(relationship_description_value_schema)
)

print(compact_domain_context)
print()
print(
    "Compact domain-context SHA-256:",
    compact_domain_context_hash,
)
print(
    "Compact characters:",
    len(compact_domain_context),
)
print(
    "Previous characters:",
    len(relationship_description_value_schema),
)
print(f"Character reduction: {reduction:.1%}")
print("Compact-context safeguards passed.")
print("No model request made.")


SQLite financial database.

Schema:
- account: account_id INTEGER PK; district_id INTEGER FK->district.district_id; frequency TEXT; date DATE.
- card: card_id INTEGER PK; disp_id INTEGER FK->disp.disp_id; type TEXT; issued DATE.
- client: client_id INTEGER PK; gender TEXT; birth_date DATE; district_id INTEGER FK->district.district_id.
- disp: disp_id INTEGER PK; client_id INTEGER FK->client.client_id; account_id INTEGER FK->account.account_id; type TEXT.
- district: district_id INTEGER PK; A2 TEXT; A3 TEXT; A4 TEXT; A5 TEXT; A6 TEXT; A7 TEXT; A8 INTEGER; A9 INTEGER; A10 REAL; A11 INTEGER; A12 REAL; A13 REAL; A14 INTEGER; A15 INTEGER; A16 INTEGER.
- loan: loan_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; amount INTEGER; duration INTEGER; payments REAL; status TEXT.
- "order": order_id INTEGER PK; account_id INTEGER FK->account.account_id; bank_to TEXT; account_to INTEGER; amount REAL; k_symbol TEXT.
- trans: trans_id INTEGER PK; account_id INTEGER FK->account.acc

In [12]:
EXPECTED_COMPACT_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)
assert compact_domain_context_hash == EXPECTED_COMPACT_CONTEXT_SHA256


## 5. Add general SQL-construction guardrails

The remaining failures were not just missing domain facts. The model selected
extra columns, referenced tables it had not joined, used ambiguous columns, and
invented filters. These rules target those general construction errors. They
deliberately contain no question IDs, answer literals, or reference-query
fragments.


In [13]:
sql_construction_guardrails = """SQL construction checks:
- Every table referenced in SELECT, WHERE, JOIN, GROUP BY, HAVING, or ORDER BY must appear in FROM or JOIN.
- Join tables only through the foreign-key relationships supplied in the schema.
- Verify that every column belongs to the table or alias used to qualify it.
- When more than one table is used, qualify every selected, filtered, grouped, and ordered column.
- Return only the columns requested by the question, in the requested order. Do not return helper, filter, ranking, or aggregate columns unless requested.
- Do not add filters or assumptions that the question does not request.
- If the question asks for different, distinct, or unique items, use DISTINCT.
- For highest or lowest, sort the relevant expression and apply LIMIT only when one result is requested.
- For a calendar year stored in YYYY-MM-DD text, use a year extraction or a correct start-inclusive/end-exclusive date range.
- Quote identifiers whenever SQLite requires quoting.
- Before returning, check that the query is executable against the supplied schema."""
guardrailed_instructions = (
    slm_config["instructions"].rstrip()
    + "\n\n"
    + sql_construction_guardrails
)

guardrailed_instructions_hash = hashlib.sha256(
    guardrailed_instructions.encode("utf-8")
).hexdigest()

print(guardrailed_instructions)
print()
print(
    "Guardrailed-instructions SHA-256:",
    guardrailed_instructions_hash,
)
print(
    "Base instruction characters:",
    len(slm_config["instructions"]),
)
print(
    "Guardrailed instruction characters:",
    len(guardrailed_instructions),
)
# The rules must remain general rather than targeting a working question.
for forbidden_fragment in (
    "account.date",
    "trans.date",
    "client.gender",
    "loan.status",
    "POPLATEK",
    "North Bohemia",
    "Question ID",
    "reference SQL",
):
    assert forbidden_fragment.casefold() not in (
        guardrailed_instructions.casefold()
    )

required_guardrails = (
    "must appear in FROM or JOIN",
    "foreign-key relationships",
    "qualify every selected",
    "only the columns requested",
    "Do not add filters",
    "use DISTINCT",
    "YYYY-MM-DD",
    "query is executable",
)

for required_guardrail in required_guardrails:
    assert required_guardrail in guardrailed_instructions

print("General guardrail safeguards passed.")
print(
    "Compact context SHA-256:",
    compact_domain_context_hash,
)
print("No model request made.")


You translate natural-language questions into SQLite SQL.

Return exactly one executable SQLite query.
Use only the tables and columns in the supplied schema.
Do not include Markdown, code fences, comments, or explanations.
Do not invent tables or columns.

SQL construction checks:
- Every table referenced in SELECT, WHERE, JOIN, GROUP BY, HAVING, or ORDER BY must appear in FROM or JOIN.
- Join tables only through the foreign-key relationships supplied in the schema.
- Verify that every column belongs to the table or alias used to qualify it.
- When more than one table is used, qualify every selected, filtered, grouped, and ordered column.
- Return only the columns requested by the question, in the requested order. Do not return helper, filter, ranking, or aggregate columns unless requested.
- Do not add filters or assumptions that the question does not request.
- If the question asks for different, distinct, or unique items, use DISTINCT.
- For highest or lowest, sort the relevant exp

In [14]:
EXPECTED_GUARDRAIL_HASH = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)
assert guardrailed_instructions_hash == EXPECTED_GUARDRAIL_HASH
print("Guardrailed instructions hash:", guardrailed_instructions_hash)


Guardrailed instructions hash: 175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509


## 6. Use one experiment runner, not one loop per prompt variant

Historical raw runs live under `data/runs/` and are intentionally local. The
helper below is the single model/evaluation path retained from the exploratory
notebook. It writes after every question, refuses duplicate question IDs, and
supports an optional output normalizer. Nothing calls it automatically:
changing `RUN_MODEL_EXPERIMENTS` is an explicit, billable action.

The frozen split is rejected by construction. Milestone 5 uses only
`working_examples`.


In [15]:
def is_executable(result: dict) -> bool:
    reason = result.get("reason") or ""
    return (
        result.get("api_error") is None
        and not reason.startswith("candidate execution failed")
        and not reason.startswith("model request failed")
    )


def summarize_results(results: list[dict]) -> dict:
    return {
        "passed": sum(result["passed"] for result in results),
        "total": len(results),
        "executable": sum(is_executable(result) for result in results),
        "failure_reasons": dict(Counter(
            result["reason"] for result in results if not result["passed"]
        )),
        "usage": {
            token_name: sum(
                result.get("usage", {}).get(token_name, 0)
                for result in results
            )
            for token_name in ("prompt_tokens", "completion_tokens", "total_tokens")
        },
    }


def save_json_atomic(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary_path.replace(path)


def build_slm_client():
    from azure.identity import AzureCliCredential, get_bearer_token_provider
    from dotenv import find_dotenv, load_dotenv
    from openai import OpenAI

    load_dotenv(find_dotenv(usecwd=True))
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    if not endpoint.endswith("/openai/v1"):
        endpoint = f"{endpoint}/openai/v1"

    credential = AzureCliCredential(tenant_id=os.environ["AZURE_TENANT_ID"])
    token_provider = get_bearer_token_provider(
        credential,
        "https://ai.azure.com/.default",
    )
    return OpenAI(base_url=f"{endpoint}/", api_key=token_provider)


def run_working_experiment(
    *,
    client,
    variant: str,
    instructions: str,
    schema_context: str,
    output_path: Path,
    normalize_output=lambda sql: (sql, False),
) -> dict:
    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite an existing run: {output_path}"
        )

    deployment_name = os.environ["AZURE_SLM_MODEL_DEPLOYMENT_NAME"]
    run = {
        "run_version": 1,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "split": "working_dev",
        "variant": variant,
        "deployment_name": deployment_name,
        "context_sha256": sha256_text(schema_context),
        "instructions_sha256": sha256_text(instructions),
        "attempts_per_question": 1,
        "results": [],
    }

    for index, example in enumerate(working_examples, start=1):
        assert example["question_id"] not in frozen_ids
        started = perf_counter()
        try:
            response, raw_sql = request_slm_sql(
                client=client,
                deployment_name=deployment_name,
                instructions=instructions,
                schema_context=schema_context,
                question=example["question"],
                max_tokens=slm_config["max_tokens"],
            )
            generated_sql, normalized = normalize_output(raw_sql)
            reference_sql, rules = make_reference_compatible(example["SQL"])
            comparison = compare_sql_results(
                conn=conn,
                reference_sql=reference_sql,
                candidate_sql=generated_sql,
            )
            usage = response.usage.model_dump() if response.usage else {}
            result = {
                "question_id": example["question_id"],
                "difficulty": example["difficulty"],
                "question": example["question"],
                "raw_generated_sql": raw_sql,
                "generated_sql": generated_sql,
                "output_envelope_normalized": normalized,
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "expected_row_count": comparison.expected_row_count,
                "candidate_row_count": comparison.candidate_row_count,
                "reference_compatibility_rules": rules,
                "response_id": response.id,
                "response_model": response.model,
                "finish_reason": response.choices[0].finish_reason,
                "usage": usage,
                "api_error": None,
            }
        except Exception as error:
            result = {
                "question_id": example["question_id"],
                "difficulty": example["difficulty"],
                "question": example["question"],
                "generated_sql": None,
                "passed": False,
                "reason": "model request failed",
                "usage": {},
                "api_error": f"{type(error).__name__}: {error}",
            }

        result["latency_seconds"] = round(perf_counter() - started, 3)
        run["results"].append(result)
        if len({row["question_id"] for row in run["results"]}) != len(run["results"]):
            raise AssertionError("Duplicate question ID in run")
        save_json_atomic(output_path, run)

        outcome = "PASS" if result["passed"] else "FAIL"
        print(
            f"[{index:02d}/{len(working_examples):02d}] "
            f"ID {example['question_id']:>3}: {outcome} — {result['reason']}"
        )

    run["summary"] = summarize_results(run["results"])
    save_json_atomic(output_path, run)
    return run


RUN_MODEL_EXPERIMENTS = False
print("Model experiments enabled:", RUN_MODEL_EXPERIMENTS)


Model experiments enabled: False


## 7. Normalize only a strict surrounding SQL fence

Four guardrailed outputs were wrapped in Markdown despite the no-fence
instruction. Removing exactly one complete outer fence is deterministic output
parsing, not semantic SQL repair. Prose, extra text, nested fences, and partial
fences are deliberately left unchanged.


In [16]:
def unwrap_single_sql_fence(text):
    """Remove one surrounding SQL fence and nothing else."""
    stripped = (text or "").strip()
    lines = stripped.splitlines()

    if len(lines) < 3:
        return stripped, False

    opening = lines[0].strip().casefold()
    closing = lines[-1].strip()

    if opening not in {
        "```",
        "```sql",
        "```sqlite",
    }:
        return stripped, False

    if closing != "```":
        return stripped, False

    inner_lines = lines[1:-1]

    # Refuse nested or multiple fenced blocks.
    if any("```" in line for line in inner_lines):
        return stripped, False

    inner_sql = "\n".join(inner_lines).strip()

    if not inner_sql:
        return stripped, False

    return inner_sql, True
assert unwrap_single_sql_fence(
    "```sql\nSELECT 1;\n```"
) == ("SELECT 1;", True)

assert unwrap_single_sql_fence(
    "SELECT 1;"
) == ("SELECT 1;", False)

assert unwrap_single_sql_fence(
    "Here is SQL:\n```sql\nSELECT 1;\n```"
)[1] is False

assert unwrap_single_sql_fence(
    "```sql\nSELECT 1;\n```\nExtra text"
)[1] is False

print("Strict SQL-envelope normalizer validated.")


Strict SQL-envelope normalizer validated.


## Reproduce the selected experiment

The historical table below is not a substitute for executable code. These cells
provide one shared path for:

1. testing one working-development example;
2. rerunning all 12 working examples;
3. summarizing a saved run; and
4. comparing all saved context stages.

Both live-call switches default to `False`, so **Run All makes no model request**.
A rerun is a reproducibility check, not a new attempt used to revise the frozen
Milestone 5 contract. Provider-default sampling can produce slightly different
results even when the prompt hashes match.


In [17]:
def evaluate_one_working_example(
    *,
    client,
    question_id: int,
    instructions: str,
    schema_context: str,
    normalize_output=unwrap_single_sql_fence,
) -> dict:
    if question_id not in working_ids:
        raise ValueError("Use a working-development question only")
    if question_id in frozen_ids:
        raise ValueError("Frozen question access is forbidden")

    example = next(
        row for row in working_examples
        if row["question_id"] == question_id
    )
    response, raw_sql = request_slm_sql(
        client=client,
        deployment_name=os.environ["AZURE_SLM_MODEL_DEPLOYMENT_NAME"],
        instructions=instructions,
        schema_context=schema_context,
        question=example["question"],
        max_tokens=slm_config["max_tokens"],
    )
    generated_sql, was_normalized = normalize_output(raw_sql)
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=generated_sql,
    )

    return {
        "question_id": question_id,
        "question": example["question"],
        "raw_generated_sql": raw_sql,
        "generated_sql": generated_sql,
        "output_envelope_normalized": was_normalized,
        "passed": comparison.equivalent,
        "reason": comparison.reason,
        "expected_row_count": comparison.expected_row_count,
        "candidate_row_count": comparison.candidate_row_count,
        "reference_compatibility_rules": rules,
        "response_id": response.id,
        "response_model": response.model,
        "usage": response.usage.model_dump() if response.usage else {},
    }


In [18]:
RUN_ONE_EXAMPLE = False
ONE_EXAMPLE_ID = 16

if RUN_ONE_EXAMPLE:
    one_example_result = evaluate_one_working_example(
        client=build_slm_client(),
        question_id=ONE_EXAMPLE_ID,
        instructions=guardrailed_instructions,
        schema_context=compact_domain_context,
    )
    print(json.dumps(one_example_result, indent=2))
else:
    print("One-example model request skipped. Set RUN_ONE_EXAMPLE=True to run it.")


One-example model request skipped. Set RUN_ONE_EXAMPLE=True to run it.


In [19]:
RUN_ALL_12_WORKING_EXAMPLES = False
manual_rerun_path = (
    DATA_DIR / "runs" /
    "ministral_3b_compact_guardrails_strict_unwrap_working_dev_manual_v2.json"
)

if RUN_ALL_12_WORKING_EXAMPLES:
    manual_rerun = run_working_experiment(
        client=build_slm_client(),
        variant="compact-guardrails-strict-unwrap-manual-v2",
        instructions=guardrailed_instructions,
        schema_context=compact_domain_context,
        output_path=manual_rerun_path,
        normalize_output=unwrap_single_sql_fence,
    )
    print(json.dumps(manual_rerun["summary"], indent=2))
else:
    print("Twelve-question model run skipped. Set RUN_ALL_12_WORKING_EXAMPLES=True to run it.")


Twelve-question model run skipped. Set RUN_ALL_12_WORKING_EXAMPLES=True to run it.


In [20]:
stage_run_paths = {
    "Bare schema": DATA_DIR / "runs" / "ministral_3b_minimal_working_dev_v1.json",
    "Relationships": DATA_DIR / "runs" / "ministral_3b_relationship_schema_working_dev_v1.json",
    "+ descriptions": DATA_DIR / "runs" / "ministral_3b_relationship_descriptions_working_dev_v1.json",
    "+ verified values": DATA_DIR / "runs" / "ministral_3b_relationship_descriptions_values_working_dev_v1.json",
    "Compact + guardrails": DATA_DIR / "runs" / "ministral_3b_compact_guardrails_working_dev_v1.json",
    "+ strict fence unwrap": DATA_DIR / "runs" / "ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json",
    "+ one repair": DATA_DIR / "runs" / "ministral_3b_compact_guardrails_one_repair_working_dev_v1.json",
}


def summarize_saved_stage(path: Path) -> dict:
    run = json.loads(path.read_text(encoding="utf-8"))
    results = run["results"]
    usage_key = "repair_usage" if "one_repair" in path.name else "usage"
    prompt_tokens = sum(
        result.get("usage", {}).get(
            "prompt_tokens",
            result.get("usage", {}).get("input_tokens", 0),
        )
        for result in results
    )
    if usage_key == "repair_usage":
        prompt_tokens += sum(
            result.get("repair_usage", {}).get("prompt_tokens", 0)
            for result in results
        )
    return {
        "passed": sum(result["passed"] for result in results),
        "total": len(results),
        "executable": sum(is_executable(result) for result in results),
        "prompt_tokens": prompt_tokens,
    }


available_stage_summaries = {
    stage: summarize_saved_stage(path)
    for stage, path in stage_run_paths.items()
    if path.exists()
}

if available_stage_summaries:
    print(f"{'Stage':<26} {'Accuracy':>10} {'Executable':>12} {'Prompt tokens':>14}")
    print("-" * 66)
    for stage, summary in available_stage_summaries.items():
        print(
            f"{stage:<26} "
            f"{summary['passed']}/{summary['total']:>7} "
            f"{summary['executable']}/{summary['total']:>9} "
            f"{summary['prompt_tokens']:>14,}"
        )
else:
    print("No local stage artifacts found; use the recorded table below.")


Stage                        Accuracy   Executable  Prompt tokens
------------------------------------------------------------------
Bare schema                1/     12 5/       12          3,650
Relationships              1/     12 8/       12         10,022
+ descriptions             1/     12 8/       12         19,034
+ verified values          2/     12 7/       12         25,070
Compact + guardrails       4/     12 6/       12         16,406
+ strict fence unwrap      4/     12 8/       12         16,406
+ one repair               4/     12 10/       12         22,449


## Working-set results

These are the recorded one-attempt results from the local run artifacts. Prompt
tokens are totals across 12 questions; the average is rounded.

| Context | Accuracy | Executable | Prompt tokens | Avg. prompt |
|---|---:|---:|---:|---:|
| Bare schema | 1/12 (8.3%) | 5/12 (41.7%) | 3,650 | 304 |
| Relationships | 1/12 (8.3%) | 8/12 (66.7%) | 10,022 | 835 |
| + descriptions | 1/12 (8.3%) | 8/12 (66.7%) | 19,034 | 1,586 |
| + verified values | 2/12 (16.7%) | 7/12 (58.3%) | 25,070 | 2,089 |
| Compact + guardrails | **4/12 (33.3%)** | 6/12 (50.0%) | 16,406 | 1,367 |
| + strict fence unwrap | **4/12 (33.3%)** | **8/12 (66.7%)** | no new calls | — |
| + one execution repair | **4/12 (33.3%)** | 10/12 (83.3%) | +6,043 | — |

The GPT-5.2 working baseline was 4/12 with 12/12 executable and 3,572 input
tokens total. Ministral therefore matched its working-set accuracy only after
grounding and guardrails, but remained less reliable and substantially less
token-efficient.


In [21]:
recorded_ablations = {
    "bare_schema": {"passed": 1, "executable": 5, "prompt_tokens": 3650},
    "relationships": {"passed": 1, "executable": 8, "prompt_tokens": 10022},
    "descriptions": {"passed": 1, "executable": 8, "prompt_tokens": 19034},
    "verified_values": {"passed": 2, "executable": 7, "prompt_tokens": 25070},
    "compact_guardrails": {"passed": 4, "executable": 6, "prompt_tokens": 16406},
    "strict_unwrap": {"passed": 4, "executable": 8, "prompt_tokens": 16406},
    "one_repair": {"passed": 4, "executable": 10, "prompt_tokens": 22449},
}

assert recorded_ablations["strict_unwrap"]["passed"] == 4
assert recorded_ablations["strict_unwrap"]["executable"] == 8
assert recorded_ablations["one_repair"]["passed"] == 4
print("Recorded ablation summary validated.")


Recorded ablation summary validated.


### Interpretation

- **Relationships helped syntax more than meaning.** Executability rose from
  5/12 to 8/12, but accuracy stayed at 1/12.
- **Descriptions decoded opaque columns but added a large prompt cost.** They
  did not improve aggregate accuracy by themselves.
- **Verified values solved real literal mistakes.** Question 5 passed once the
  model saw `M`, `north Bohemia`, and `A11`, but the expanded prompt became too
  large and construction errors remained.
- **Compression plus general guardrails was the meaningful gain.** Accuracy
  reached 4/12 without any example-specific hints.
- **Fence unwrapping is worthwhile.** It lifted executability from 6/12 to 8/12
  with no new inference or semantic changes.
- **Execution repair is not worthwhile at this checkpoint.** It lifted
  executability to 10/12 but added 6,495 total tokens and did not recover one
  correct answer.

Failure inspection also showed that many remaining errors are semantic, not
merely syntactic: wrong join paths, wrong requested columns, incorrect date
interpretation, and invented business logic. More schema prose alone is
unlikely to solve them.


## Rejected ablation: one execution-error repair

The repair contract is retained for provenance, but the repair loop itself is
removed from this notebook. It was triggered only by SQLite execution errors,
received no reference SQL or expected result, and allowed one extra call. The
result was 4/12 accuracy—the same as strict unwrapping—at a substantial token
cost. Keeping it in the selected pipeline would optimize executability rather
than task correctness.


In [22]:
repair_instructions = (
    """You repair a failed SQLite query.

Return exactly one executable SQLite query.
Preserve the original question's intended answer.
Use only the supplied tables, columns, relationships, meanings, and values.
Fix the execution error without adding unrelated logic.
Do not include Markdown, code fences, comments, or explanations."""
    + "\n\n"
    + sql_construction_guardrails
)

repair_user_template = """Schema and domain context:
{schema_context}

Original question:
{question}

Failed SQL:
{failed_sql}

SQLite execution error:
{execution_error}

Return the repaired SQLite query only."""
repair_contract = {
    "instructions": repair_instructions,
    "user_template": repair_user_template,
    "trigger": "candidate execution failure only",
    "maximum_repairs": 1,
    "reference_sql_available": False,
    "expected_results_available": False,
    "output_normalization": (
        "strict single SQL-fence unwrapping"
    ),
}

repair_contract_json = json.dumps(
    repair_contract,
    sort_keys=True,
    separators=(",", ":"),
)

repair_contract_sha256 = hashlib.sha256(
    repair_contract_json.encode("utf-8")
).hexdigest()

print("Repair-contract SHA-256:", repair_contract_sha256)
print(
    "Repair instruction characters:",
    len(repair_instructions),
)
print(
    "Repair template characters:",
    len(repair_user_template),
)


Repair-contract SHA-256: 64837f617470bc7eb388e9d429f8fd2cf78abee177249bda2730ff586d326e24
Repair instruction characters: 1391
Repair template characters: 183


In [23]:
EXPECTED_REPAIR_CONTRACT_SHA256 = (
    "64837f617470bc7eb388e9d429f8fd2cf78abee177249bda2730ff586d326e24"
)
assert repair_contract_sha256 == EXPECTED_REPAIR_CONTRACT_SHA256


## Frozen Milestone 5 contract

The experiment has now answered the context-engineering question. The candidate
to carry forward is fixed by the hashes below. Do not add the repair pass, do
not inspect frozen examples while editing the prompt, and do not rerun earlier
ablations merely to obtain a luckier sample.

The next engineering step is to move this exact context, instruction extension,
and strict envelope normalizer into reusable tested code. Only after that code
is validated should the 24-question frozen evaluation be run once and sealed.


In [24]:
milestone_5_contract = {
    "model": "ministral-3b-2410",
    "selection_split": "working_dev",
    "selection_questions": 12,
    "frozen_questions_used_for_selection": 0,
    "context": "compact financial-domain context",
    "context_sha256": EXPECTED_COMPACT_CONTEXT_SHA256,
    "instructions": "baseline instructions plus general SQL guardrails",
    "instructions_sha256": EXPECTED_GUARDRAIL_HASH,
    "output_normalization": "strict single surrounding SQL-fence unwrap",
    "candidate_sql_repair": False,
    "attempts_per_question": 1,
    "working_accuracy": "4/12",
    "working_executable": "8/12",
}

assert milestone_5_contract["frozen_questions_used_for_selection"] == 0
assert milestone_5_contract["candidate_sql_repair"] is False
assert sha256_text(compact_domain_context) == milestone_5_contract["context_sha256"]
assert sha256_text(guardrailed_instructions) == milestone_5_contract["instructions_sha256"]

print(json.dumps(milestone_5_contract, indent=2))
print("No model request made.")


{
  "model": "ministral-3b-2410",
  "selection_split": "working_dev",
  "selection_questions": 12,
  "frozen_questions_used_for_selection": 0,
  "context": "compact financial-domain context",
  "context_sha256": "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987",
  "instructions": "baseline instructions plus general SQL guardrails",
  "instructions_sha256": "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509",
  "output_normalization": "strict single surrounding SQL-fence unwrap",
  "candidate_sql_repair": false,
  "attempts_per_question": 1,
  "working_accuracy": "4/12",
  "working_executable": "8/12"
}
No model request made.
